In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import ipaddress
import re

In [53]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import average_precision_score
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import average_precision_score
from lightgbm import LGBMClassifier
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, precision_recall_curve, auc
from lightgbm import LGBMClassifier

In [6]:
csv_path = Path("/content/drive/MyDrive/iot/iot device name/network/packet_csvs_device_only/all_pcaps_device_rows_merged.csv")

In [39]:
df = pd.read_csv(csv_path, low_memory=False)

In [40]:
print("raw shape:", df.shape)
print(df.columns.tolist())
display(df.head())

raw shape: (214040, 19)
['frame_time_epoch', 'src_ip', 'dst_ip', 'ip_proto', 'tcp_srcport', 'tcp_dstport', 'udp_srcport', 'udp_dstport', 'frame_len', 'tcp_flags', 'ip_dsfield', 'tcp_stream', 'udp_stream', 'mqtt_topic', 'info', 'device_name', 'mapping_source', 'mapping_confidence', 'pcap_name']


,frame_time_epoch,src_ip,dst_ip,ip_proto,tcp_srcport,tcp_dstport,udp_srcport,udp_dstport,frame_len,tcp_flags,ip_dsfield,tcp_stream,udp_stream,mqtt_topic,info,device_name,mapping_source,mapping_confidence,pcap_name
0,1.554324e+09,192.168.1.152,3.122.49.24,6,52976,1883,NaN,NaN,1516,0x0010,0x02,1,NaN,/iot/locator,"Publish Message (id=12526) [/iot/locator], Pub...",IoT_Thermostat,topic,high,normal_10.pcap
1,1.554324e+09,192.168.1.152,3.122.49.24,6,52976,1883,NaN,NaN,1516,0x0010,0x02,1,NaN,/smarthome/thermostat,Publish Message (id=12535) [/smarthome/thermos...,IoT_Thermostat,topic,high,normal_10.pcap
2,1.554324e+09,192.168.1.152,3.122.49.24,6,52976,1883,NaN,NaN,523,0x0018,0x02,1,NaN,/smarthome/garageDoor,Publish Message (id=12546) [/smarthome/garageD...,IoT_Thermostat,topic,high,normal_10.pcap
3,1.554324e+09,192.168.1.152,3.122.49.24,6,52976,1883,NaN,NaN,223,0x0018,0x02,1,NaN,/smarthome/motionLights,Publish Message (id=12549) [/smarthome/motionL...,IoT_Motion_Light,topic,high,normal_10.pcap
4,1.554324e+09,192.168.1.152,3.122.49.24,6,52976,1883,NaN,NaN,350,0x0018,0x02,1,NaN,/smarfactory/Modbus-data,Publish Message (id=12550) [/smarfactory/Modbu...,IoT_Motion_Light,topic,high,normal_10.pcap


In [41]:
work = df.copy()

In [ ]:
source_candidates = ["source", "pcap", "file", "source_file", "pcap_file", "trace"]
source_col = None
for c in source_candidates:
    if c in work.columns:
        source_col = c
        break

if source_col is None:
    work["_pcap_group"] = "single_source"
    source_col = "_pcap_group"

# device_name is reserved; empty values are recorded as UNKNOWN_DEVICE
work["device_name"] = work["device_name"].fillna("UNKNOWN_DEVICE").astype(str).str.strip()
work.loc[work["device_name"] == "", "device_name"] = "UNKNOWN_DEVICE"

In [ ]:
num_defaults = {
    "frame_time_epoch": 0.0,
    "frame_len": 0.0,
    "ip_proto": -1,
    "tcp_srcport": -1,
    "tcp_dstport": -1,
    "udp_srcport": -1,
    "udp_dstport": -1,
    "tcp_stream": -1,
    "udp_stream": -1,
}

for c, default in num_defaults.items():
    if c in work.columns:
        work[c] = pd.to_numeric(work[c], errors="coerce").fillna(default)
    else:
        work[c] = default

for c in ["src_ip", "dst_ip", "tcp_flags", "ip_dsfield", "mqtt_topic", source_col]:
    if c in work.columns:
        work[c] = work[c].fillna("UNK").astype(str)
    else:
        work[c] = "UNK"


work["src_port"] = work["tcp_srcport"].where(work["tcp_srcport"] != -1, work["udp_srcport"])
work["dst_port"] = work["tcp_dstport"].where(work["tcp_dstport"] != -1, work["udp_dstport"])
work["src_port"] = work["src_port"].fillna(-1).astype(int)
work["dst_port"] = work["dst_port"].fillna(-1).astype(int)
work["ip_proto"] = work["ip_proto"].fillna(-1).astype(int)


In [44]:
pcap_series = work["pcap_name"].astype(str)

In [45]:
default_flow_id = (
    pcap_series
    + "::P" + work["ip_proto"].astype(str)
    + "::" + work["src_ip"]
    + "::" + work["src_port"].astype(str)
    + "::" + work["dst_ip"]
    + "::" + work["dst_port"].astype(str)
)

work["flow_id"] = default_flow_id

tcp_mask = (work["ip_proto"] == 6) & (work["tcp_stream"] != -1)
udp_mask = (work["ip_proto"] == 17) & (work["udp_stream"] != -1)

work.loc[tcp_mask, "flow_id"] = (
    pcap_series[tcp_mask]
    + "::TCPSTREAM::"
    + work.loc[tcp_mask, "tcp_stream"].astype(int).astype(str)
)

work.loc[udp_mask, "flow_id"] = (
    pcap_series[udp_mask]
    + "::UDPSTREAM::"
    + work.loc[udp_mask, "udp_stream"].astype(int).astype(str)
)

print("rows:", len(work))
print("unique flow_id:", work["flow_id"].nunique())

rows: 214040
unique flow_id: 14


In [46]:
def parse_flag(x):
    s = str(x).strip()
    if s == "" or s.lower() == "nan" or s == "UNK":
        return 0
    try:
        return int(s, 16) if s.lower().startswith("0x") else int(float(s))
    except:
        return 0

work["tcp_flags_int"] = work["tcp_flags"].apply(parse_flag)

def bitwise_or(series):
    out = 0
    for v in series.astype(int).tolist():
        out |= v
    return out

In [47]:
flow_df = (
    work.groupby(["device_name", "flow_id"], as_index=False, dropna=False)
        .agg(
            first_time=("frame_time_epoch", "min"),
            last_time=("frame_time_epoch", "max"),
            IN_BYTES=("frame_len", "sum"),
            IN_PKTS=("frame_len", "size"),
            IPV4_DST_ADDR=("dst_ip", "first"),
            L4_DST_PORT=("dst_port", "first"),
            L4_SRC_PORT=("src_port", "first"),
            PROTOCOL=("ip_proto", "first"),
            SRC_TOS=("ip_dsfield", "first"),
            TCP_FLAGS=("tcp_flags_int", bitwise_or),
            PCAP_GROUP=("pcap_name", "first"),
            MQTT_TOPIC=("mqtt_topic", "first"),
        )
)

flow_df["DURATION"] = flow_df["last_time"] - flow_df["first_time"]
flow_df["DURATION"] = pd.to_numeric(flow_df["DURATION"], errors="coerce").fillna(0.0)

def to_ipv4_prefix(x):
    try:
        ip = ipaddress.ip_address(str(x))
        if ip.version == 4:
            p = str(x).split(".")
            return ".".join(p[:3]) + ".0/24"
        return str(x)
    except:
        return "UNK"

flow_df["IPV4_DST_NET"] = flow_df["IPV4_DST_ADDR"].apply(to_ipv4_prefix)

print("flow_df shape:", flow_df.shape)
print("\nflow counts per device:")
print(flow_df["device_name"].value_counts())

display(flow_df.head())

flow_out = csv_path.with_name("meid20_flow_table_from_packets.csv")
flow_df.to_csv(flow_out, index=False, encoding="utf-8-sig")
print("saved:", flow_out)

flow_df shape: (87, 16)

flow counts per device:
device_name
IoT_Modbus          14
IoT_Motion_Light    13
IoT_Fridge          13
IoT_Thermostat      13
IoT_Weather         13
IoT_Garage_Door     12
IoT_GPS_Tracker      9
Name: count, dtype: int64


,device_name,flow_id,first_time,last_time,IN_BYTES,IN_PKTS,IPV4_DST_ADDR,L4_DST_PORT,L4_SRC_PORT,PROTOCOL,SRC_TOS,TCP_FLAGS,PCAP_GROUP,MQTT_TOPIC,DURATION,IPV4_DST_NET
0,IoT_Fridge,normal_1.pcap::TCPSTREAM::3,1.554220e+09,1.554230e+09,789834,1928,3.122.49.24,1883,52976,6,0x02,24,normal_1.pcap,/smarthome/garageDoor,9458.245741,3.122.49.0/24
1,IoT_Fridge,normal_10.pcap::TCPSTREAM::1,1.554327e+09,1.554334e+09,1639,9,3.122.49.24,1883,52976,6,0x02,24,normal_10.pcap,/smarthome/garageDoor,7165.339273,3.122.49.0/24
2,IoT_Fridge,normal_11.pcap::TCPSTREAM::0,1.554343e+09,1.554343e+09,149,1,3.122.49.24,1883,52976,6,0x02,24,normal_11.pcap,/smarthome/fridge,0.000000,3.122.49.0/24
3,IoT_Fridge,normal_12.pcap::TCPSTREAM::0,1.554351e+09,1.554352e+09,1662,2,3.122.49.24,1883,52976,6,0x02,24,normal_12.pcap,/smarthome/fridge,590.085667,3.122.49.0/24
4,IoT_Fridge,normal_13.pcap::TCPSTREAM::1,1.554274e+09,1.554358e+09,34580,66,3.122.49.24,1883,52976,6,0x02,24,normal_13.pcap,/smarthome/fridge,84351.586651,3.122.49.0/24


saved: /content/drive/MyDrive/iot/iot device name/network/packet_csvs_device_only/meid20_flow_table_from_packets.csv


In [48]:
flow_path = Path("/content/drive/MyDrive/iot/iot device name/network/packet_csvs_device_only/meid20_flow_table_from_packets.csv")
flow_df = pd.read_csv(flow_path, low_memory=False)

print("flow_df shape:", flow_df.shape)
print(flow_df["device_name"].value_counts())

work = flow_df.copy()


flow_df shape: (87, 16)
device_name
IoT_Modbus          14
IoT_Motion_Light    13
IoT_Fridge          13
IoT_Thermostat      13
IoT_Weather         13
IoT_Garage_Door     12
IoT_GPS_Tracker      9
Name: count, dtype: int64


In [ ]:
# Numeric column
for c in ["DURATION", "IN_BYTES", "IN_PKTS"]:
    work[c] = pd.to_numeric(work[c], errors="coerce").fillna(0.0)

# Category column
for c in ["IPV4_DST_NET", "L4_DST_PORT", "L4_SRC_PORT", "PROTOCOL", "SRC_TOS", "TCP_FLAGS"]:
    work[c] = work[c].fillna("UNK").astype(str)

work["device_name"] = work["device_name"].fillna("UNKNOWN_DEVICE").astype(str).str.strip()
work.loc[work["device_name"] == "", "device_name"] = "UNKNOWN_DEVICE"

feature_cols = [
    "DURATION",
    "IN_BYTES",
    "IN_PKTS",
    "IPV4_DST_NET",
    "L4_DST_PORT",
    "L4_SRC_PORT",
    "PROTOCOL",
    "SRC_TOS",
    "TCP_FLAGS",
]

X = work[feature_cols].copy()
y = work["device_name"].copy()

print("\nclass counts:")
print(y.value_counts())

categorical_cols = ["IPV4_DST_NET", "L4_DST_PORT", "L4_SRC_PORT", "PROTOCOL", "SRC_TOS", "TCP_FLAGS"]
numeric_cols = ["DURATION", "IN_BYTES", "IN_PKTS"]

try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=True)

preprocess = ColumnTransformer(
    transformers=[
        ("cat", ohe, categorical_cols),
        ("num", "passthrough", numeric_cols),
    ],
    remainder="drop"
)


class counts:
device_name
IoT_Modbus          14
IoT_Motion_Light    13
IoT_Fridge          13
IoT_Thermostat      13
IoT_Weather         13
IoT_Garage_Door     12
IoT_GPS_Tracker      9
Name: count, dtype: int64


In [ ]:

for c in ["DURATION", "IN_BYTES", "IN_PKTS"]:
    work[c] = pd.to_numeric(work[c], errors="coerce").fillna(0.0)


for c in ["IPV4_DST_NET", "L4_DST_PORT", "L4_SRC_PORT", "PROTOCOL", "SRC_TOS", "TCP_FLAGS"]:
    work[c] = work[c].fillna("UNK").astype(str)

work["device_name"] = work["device_name"].fillna("UNKNOWN_DEVICE").astype(str).str.strip()
work.loc[work["device_name"] == "", "device_name"] = "UNKNOWN_DEVICE"

feature_cols = [
    "DURATION",
    "IN_BYTES",
    "IN_PKTS",
    "IPV4_DST_NET",
    "L4_DST_PORT",
    "L4_SRC_PORT",
    "PROTOCOL",
    "SRC_TOS",
    "TCP_FLAGS",
]

X = work[feature_cols].copy()
y = work["device_name"].copy()

print("\nclass counts:")
print(y.value_counts())

categorical_cols = ["IPV4_DST_NET", "L4_DST_PORT", "L4_SRC_PORT", "PROTOCOL", "SRC_TOS", "TCP_FLAGS"]
numeric_cols = ["DURATION", "IN_BYTES", "IN_PKTS"]

try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=True)

preprocess = ColumnTransformer(
    transformers=[
        ("cat", ohe, categorical_cols),
        ("num", "passthrough", numeric_cols),
    ],
    remainder="drop"
)


class counts:
device_name
IoT_Modbus          14
IoT_Motion_Light    13
IoT_Fridge          13
IoT_Thermostat      13
IoT_Weather         13
IoT_Garage_Door     12
IoT_GPS_Tracker      9
Name: count, dtype: int64


In [54]:
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

rows = []

for cls in sorted(y.unique()):
    y_bin = (y == cls).astype(int)

    fold_aps = []
    fold_aucprs = []

    for train_idx, test_idx in skf.split(X, y):
        X_train = X.iloc[train_idx]
        X_test  = X.iloc[test_idx]
        y_train_bin = y_bin.iloc[train_idx]
        y_test_bin  = y_bin.iloc[test_idx]

        if y_train_bin.sum() == 0 or y_test_bin.sum() == 0:
            continue

        model = Pipeline([
            ("prep", preprocess),
            ("clf", LGBMClassifier(
                objective="binary",
                class_weight="balanced",
                n_estimators=300,
                learning_rate=0.05,
                num_leaves=31,
                subsample=0.9,
                colsample_bytree=0.9,
                random_state=42,
                n_jobs=-1
            ))
        ])

        model.fit(X_train, y_train_bin)
        y_score = model.predict_proba(X_test)[:, 1]

        ap = average_precision_score(y_test_bin, y_score)

        precision, recall, _ = precision_recall_curve(y_test_bin, y_score)
        aucpr = auc(recall, precision)

        fold_aps.append(ap)
        fold_aucprs.append(aucpr)

    rows.append({
        "device_name": cls,
        "AP_mean_cv": np.mean(fold_aps) if fold_aps else np.nan,
        "AUCPR_mean_cv": np.mean(fold_aucprs) if fold_aucprs else np.nan,
        "num_folds_used": len(fold_aps),
        "total_pos": int(y_bin.sum())
    })

[LightGBM] [Info] Number of positive: 8, number of negative: 50
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.014719 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 61
[LightGBM] [Info] Number of data points in the train set: 58, number of used features: 3
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 9, number of negative: 49
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000262 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 61
[LightGBM] [Info] Number of data points in the train set: 58, number of used features: 3
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightG

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 6, number of negative: 52
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.015788 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 61
[LightGBM] [Info] Number of data points in the train set: 58, number of used features: 3
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -

...

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 9, number of negative: 49
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000023 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 61
[LightGBM] [Info] Number of data points in the train set: 58, number of used features: 3
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 9, number of negative: 49
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000033 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 61
[LightGBM] [Info] Number of data points in the train set: 58, number of used features: 3
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [55]:
result_df = pd.DataFrame(rows).sort_values("AUCPR_mean_cv", ascending=False)

macro_ap = result_df["AP_mean_cv"].dropna().mean()
weighted_ap = np.average(result_df["AP_mean_cv"].fillna(0), weights=result_df["total_pos"])

macro_aucpr = result_df["AUCPR_mean_cv"].dropna().mean()
weighted_aucpr = np.average(result_df["AUCPR_mean_cv"].fillna(0), weights=result_df["total_pos"])

print("\n==============================")
print(f"Macro mean CV AP       = {macro_ap:.6f}")
print(f"Weighted mean CV AP    = {weighted_ap:.6f}")
print(f"Macro mean CV AUCPR    = {macro_aucpr:.6f}")
print(f"Weighted mean CV AUCPR = {weighted_aucpr:.6f}")
print("==============================")

display(result_df)

out_path = flow_path.with_name("meid20_cv_results_3fold.csv")
result_df.to_csv(out_path, index=False, encoding="utf-8-sig")
print("saved:", out_path)


Macro mean CV AP       = 0.535804
Weighted mean CV AP    = 0.542366
Macro mean CV AUCPR    = 0.504317
Weighted mean CV AUCPR = 0.515038


,device_name,AP_mean_cv,AUCPR_mean_cv,num_folds_used,total_pos
5,IoT_Thermostat,0.865079,0.857978,3,13
6,IoT_Weather,0.792582,0.801208,3,13
3,IoT_Modbus,0.631962,0.698861,3,14
1,IoT_GPS_Tracker,0.475092,0.394292,3,9
0,IoT_Fridge,0.423886,0.365138,3,13
4,IoT_Motion_Light,0.258135,0.206470,3,13
2,IoT_Garage_Door,0.303888,0.206271,3,12


saved: /content/drive/MyDrive/iot/iot device name/network/packet_csvs_device_only/meid20_cv_results_3fold.csv


In [62]:
import pandas as pd
from pathlib import Path

flow_path = Path("/content/drive/MyDrive/iot/iot device name/network/packet_csvs_device_only/meid20_flow_table_from_packets.csv")
flow_df = pd.read_csv(flow_path, low_memory=False)

pcap_time = (
    flow_df.groupby("PCAP_GROUP", as_index=False)
    .agg(
        start_time=("first_time", "min"),
        end_time=("last_time", "max"),
        n_flows=("flow_id", "count")
    )
    .sort_values("start_time")
)

pcap_time["start_dt"] = pd.to_datetime(pcap_time["start_time"], unit="s")
pcap_time["end_dt"] = pd.to_datetime(pcap_time["end_time"], unit="s")

display(pcap_time)

,PCAP_GROUP,start_time,end_time,n_flows,start_dt,end_dt
0,normal_1.pcap,1.554220e+09,1.554230e+09,7,2019-04-02 15:52:05.047519922,2019-04-02 18:35:25.950786114
5,normal_2.pcap,1.554230e+09,1.554241e+09,7,2019-04-02 18:35:26.663904905,2019-04-02 21:36:42.551315069
6,normal_3.pcap,1.554241e+09,1.554252e+09,7,2019-04-02 21:36:42.879638910,2019-04-03 00:42:29.332081079
7,normal_4.pcap,1.554252e+09,1.554263e+09,7,2019-04-03 00:42:59.370634079,2019-04-03 03:45:15.288013935
8,normal_5.pcap,1.554263e+09,1.554274e+09,7,2019-04-03 03:45:16.288305044,2019-04-03 06:47:20.014173985
4,normal_13.pcap,1.554274e+09,1.554364e+09,7,2019-04-03 06:47:21.026434898,2019-04-04 07:40:26.510544062
9,normal_6.pcap,1.554279e+09,1.554288e+09,6,2019-04-03 08:09:45.828012943,2019-04-03 10:33:34.292262077
10,normal_7.pcap,1.554290e+09,1.554301e+09,7,2019-04-03 11:11:22.508232117,2019-04-03 14:19:43.618556023
11,normal_8.pcap,1.554301e+09,1.554308e+09,7,2019-04-03 14:19:43.947191954,2019-04-03 16:18:38.254220963
12,normal_9.pcap,1.554312e+09,1.554324e+09,7,2019-04-03 17:26:53.651465893,2019-04-03 20:38:05.385795116


In [64]:
import pandas as pd
import numpy as np
from pathlib import Path

flow_path = Path("/content/drive/MyDrive/iot/iot device name/network/packet_csvs_device_only/meid20_flow_table_from_packets.csv")
flow_df = pd.read_csv(flow_path, low_memory=False)

In [65]:
print("flow_df shape:", flow_df.shape)
display(flow_df.head())

work = flow_df.copy()


flow_df shape: (87, 16)


,device_name,flow_id,first_time,last_time,IN_BYTES,IN_PKTS,IPV4_DST_ADDR,L4_DST_PORT,L4_SRC_PORT,PROTOCOL,SRC_TOS,TCP_FLAGS,PCAP_GROUP,MQTT_TOPIC,DURATION,IPV4_DST_NET
0,IoT_Fridge,normal_1.pcap::TCPSTREAM::3,1.554220e+09,1.554230e+09,789834,1928,3.122.49.24,1883,52976,6,0x02,24,normal_1.pcap,/smarthome/garageDoor,9458.245741,3.122.49.0/24
1,IoT_Fridge,normal_10.pcap::TCPSTREAM::1,1.554327e+09,1.554334e+09,1639,9,3.122.49.24,1883,52976,6,0x02,24,normal_10.pcap,/smarthome/garageDoor,7165.339273,3.122.49.0/24
2,IoT_Fridge,normal_11.pcap::TCPSTREAM::0,1.554343e+09,1.554343e+09,149,1,3.122.49.24,1883,52976,6,0x02,24,normal_11.pcap,/smarthome/fridge,0.000000,3.122.49.0/24
3,IoT_Fridge,normal_12.pcap::TCPSTREAM::0,1.554351e+09,1.554352e+09,1662,2,3.122.49.24,1883,52976,6,0x02,24,normal_12.pcap,/smarthome/fridge,590.085667,3.122.49.0/24
4,IoT_Fridge,normal_13.pcap::TCPSTREAM::1,1.554274e+09,1.554358e+09,34580,66,3.122.49.24,1883,52976,6,0x02,24,normal_13.pcap,/smarthome/fridge,84351.586651,3.122.49.0/24


In [79]:
work = flow_df.copy()

# 用 flow 中点时间
work["mid_time"] = (work["first_time"] + work["last_time"]) / 2.0
work["mid_dt"] = pd.to_datetime(work["mid_time"], unit="s")

work["hour"] = work["mid_dt"].dt.hour
work["minute"] = work["mid_dt"].dt.minute

# idle = 02:00-06:00
# active = 其余时间
def assign_mode(hour, minute):
    hm = hour * 60 + minute
    idle = (2 * 60) <= hm < (6 * 60)
    return "idle" if idle else "active"

work["mode"] = [assign_mode(h, m) for h, m in zip(work["hour"], work["minute"])]

print(work["mode"].value_counts())
display(work[["PCAP_GROUP", "device_name", "first_time", "last_time", "mid_dt", "mode"]].head(30))

mode
active    68
idle      19
Name: count, dtype: int64


,PCAP_GROUP,device_name,first_time,last_time,mid_dt,mode
0,normal_1.pcap,IoT_Fridge,1.554220e+09,1.554230e+09,2019-04-02 17:10:56.552106380,active
1,normal_10.pcap,IoT_Fridge,1.554327e+09,1.554334e+09,2019-04-03 22:31:09.387634516,active
2,normal_11.pcap,IoT_Fridge,1.554343e+09,1.554343e+09,2019-04-04 01:52:08.281894922,active
3,normal_12.pcap,IoT_Fridge,1.554351e+09,1.554352e+09,2019-04-04 04:16:49.589401484,idle
4,normal_13.pcap,IoT_Fridge,1.554274e+09,1.554358e+09,2019-04-03 18:30:24.844794512,active
5,normal_2.pcap,IoT_Fridge,1.554230e+09,1.554240e+09,2019-04-02 20:01:21.434290409,active
6,normal_3.pcap,IoT_Fridge,1.554242e+09,1.554252e+09,2019-04-02 23:17:30.859336853,active
7,normal_4.pcap,IoT_Fridge,1.554252e+09,1.554263e+09,2019-04-03 02:14:48.139443398,idle
8,normal_5.pcap,IoT_Fridge,1.554263e+09,1.554274e+09,2019-04-03 05:16:20.528744936,idle
9,normal_6.pcap,IoT_Fridge,1.554279e+09,1.554287e+09,2019-04-03 09:17:19.702805042,active


In [80]:
idle_df = work[work["mode"] == "idle"].copy()
active_df = work[work["mode"] == "active"].copy()

train_mix = pd.concat([idle_df, active_df], axis=0).reset_index(drop=True)
test_idle = idle_df.reset_index(drop=True)
test_active = active_df.reset_index(drop=True)
test_mix = pd.concat([idle_df, active_df], axis=0).reset_index(drop=True)

print("train_mix:", train_mix.shape)
print("test_idle:", test_idle.shape)
print("test_active:", test_active.shape)
print("test_mix:", test_mix.shape)

print("\nIdle class counts:")
print(test_idle["device_name"].value_counts())

print("\nActive class counts:")
print(test_active["device_name"].value_counts())

train_mix: (87, 21)
test_idle: (19, 21)
test_active: (68, 21)
test_mix: (87, 21)

Idle class counts:
device_name
IoT_Fridge          3
IoT_Motion_Light    3
IoT_Modbus          3
IoT_Thermostat      3
IoT_Weather         3
IoT_Garage_Door     2
IoT_GPS_Tracker     2
Name: count, dtype: int64

Active class counts:
device_name
IoT_Modbus          11
IoT_Garage_Door     10
IoT_Fridge          10
IoT_Thermostat      10
IoT_Motion_Light    10
IoT_Weather         10
IoT_GPS_Tracker      7
Name: count, dtype: int64


In [81]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import precision_recall_curve, auc, average_precision_score
from lightgbm import LGBMClassifier

# =========================================================
# 1) 读取 flow 表
# =========================================================
flow_path = Path("/content/drive/MyDrive/iot/iot device name/network/packet_csvs_device_only/meid20_flow_table_from_packets.csv")
flow_df = pd.read_csv(flow_path, low_memory=False)

print("flow_df shape:", flow_df.shape)
print(flow_df["device_name"].value_counts())

work = flow_df.copy()

# =========================================================
# 2) 用 flow 中点时间做全局时间排序
# =========================================================
work["mid_time"] = (work["first_time"] + work["last_time"]) / 2.0
work = work.sort_values("mid_time").reset_index(drop=True)

# 前20% = idle，后80% = active
idle_cut_idx = int(np.floor(0.2 * len(work)))
idle_cut_idx = max(idle_cut_idx, 1)  # 至少留 1 个

work["mode"] = "active"
work.loc[:idle_cut_idx - 1, "mode"] = "idle"

print("\nmode counts:")
print(work["mode"].value_counts())

display(work[["device_name", "first_time", "last_time", "mid_time", "mode"]].head(10))
display(work[["device_name", "first_time", "last_time", "mid_time", "mode"]].tail(10))

# =========================================================
# 3) 在每个 mode 内再按时间顺序切分：
#    前70%做 train，后30%做 test
# =========================================================
def temporal_split(df, train_ratio=0.7):
    df = df.sort_values("mid_time").reset_index(drop=True)
    cut = int(np.floor(train_ratio * len(df)))
    cut = max(cut, 1)
    if cut >= len(df):
        cut = len(df) - 1
    train_df = df.iloc[:cut].copy()
    test_df = df.iloc[cut:].copy()
    return train_df, test_df

idle_df = work[work["mode"] == "idle"].copy()
active_df = work[work["mode"] == "active"].copy()

if len(idle_df) < 2 or len(active_df) < 2:
    raise ValueError("idle 或 active 样本太少，无法再做时间切分。")

idle_train, idle_test = temporal_split(idle_df, train_ratio=0.7)
active_train, active_test = temporal_split(active_df, train_ratio=0.7)

train_mix = pd.concat([idle_train, active_train], axis=0).reset_index(drop=True)
test_idle = idle_test.reset_index(drop=True)
test_active = active_test.reset_index(drop=True)
test_mix = pd.concat([idle_test, active_test], axis=0).reset_index(drop=True)

print("\ntrain_mix shape:", train_mix.shape)
print("test_idle shape:", test_idle.shape)
print("test_active shape:", test_active.shape)
print("test_mix shape:", test_mix.shape)

print("\ntrain_mix class counts:")
print(train_mix["device_name"].value_counts())

print("\ntest_idle class counts:")
print(test_idle["device_name"].value_counts())

print("\ntest_active class counts:")
print(test_active["device_name"].value_counts())

# =========================================================
# 4) 特征设置
# =========================================================
USE_MQTT_TOPIC = False

feature_cols = [
    "DURATION",
    "IN_BYTES",
    "IN_PKTS",
    "IPV4_DST_NET",
    "L4_DST_PORT",
    "L4_SRC_PORT",
    "PROTOCOL",
    "SRC_TOS",
    "TCP_FLAGS",
]

categorical_cols = [
    "IPV4_DST_NET",
    "L4_DST_PORT",
    "L4_SRC_PORT",
    "PROTOCOL",
    "SRC_TOS",
    "TCP_FLAGS",
]

numeric_cols = [
    "DURATION",
    "IN_BYTES",
    "IN_PKTS",
]

if USE_MQTT_TOPIC:
    feature_cols.append("MQTT_TOPIC")
    categorical_cols.append("MQTT_TOPIC")

# =========================================================
# 5) 基础整理
# =========================================================
def prepare_df(df):
    out = df.copy()

    out["device_name"] = out["device_name"].fillna("UNKNOWN_DEVICE").astype(str).str.strip()
    out.loc[out["device_name"] == "", "device_name"] = "UNKNOWN_DEVICE"

    for c in numeric_cols:
        out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0.0)

    for c in categorical_cols:
        out[c] = out[c].fillna("UNK").astype(str).str.strip()
        out.loc[out[c] == "", c] = "UNK"

    return out

train_mix_p = prepare_df(train_mix)
test_idle_p = prepare_df(test_idle)
test_active_p = prepare_df(test_active)
test_mix_p = prepare_df(test_mix)

# =========================================================
# 6) 预处理器
# =========================================================
try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=True)

preprocess = ColumnTransformer(
    transformers=[
        ("cat", ohe, categorical_cols),
        ("num", "passthrough", numeric_cols),
    ],
    remainder="drop"
)

# =========================================================
# 7) one-vs-rest AUCPR
# =========================================================
def evaluate_ovr_aucpr(train_df, test_df):
    X_train = train_df[feature_cols].copy()
    y_train = train_df["device_name"].copy()

    X_test = test_df[feature_cols].copy()
    y_test = test_df["device_name"].copy()

    common_classes = sorted(set(y_train.unique()) & set(y_test.unique()))
    rows = []

    for cls in common_classes:
        y_train_bin = (y_train == cls).astype(int)
        y_test_bin = (y_test == cls).astype(int)

        if y_train_bin.sum() == 0 or y_test_bin.sum() == 0:
            continue

        model = Pipeline([
            ("prep", preprocess),
            ("clf", LGBMClassifier(
                objective="binary",
                class_weight="balanced",
                n_estimators=300,
                learning_rate=0.05,
                num_leaves=31,
                subsample=0.9,
                colsample_bytree=0.9,
                random_state=42,
                n_jobs=-1
            ))
        ])

        model.fit(X_train, y_train_bin)
        y_score = model.predict_proba(X_test)[:, 1]

        ap = average_precision_score(y_test_bin, y_score)
        precision, recall, _ = precision_recall_curve(y_test_bin, y_score)
        aucpr = auc(recall, precision)

        rows.append({
            "device_name": cls,
            "AP": ap,
            "AUCPR": aucpr,
            "train_pos": int(y_train_bin.sum()),
            "test_pos": int(y_test_bin.sum())
        })

    res = pd.DataFrame(rows)

    if len(res) == 0:
        return res, {
            "macro_ap": np.nan,
            "weighted_ap": np.nan,
            "macro_aucpr": np.nan,
            "weighted_aucpr": np.nan,
        }

    res = res.sort_values("AUCPR", ascending=False)

    return res, {
        "macro_ap": res["AP"].mean(),
        "weighted_ap": np.average(res["AP"], weights=res["test_pos"]),
        "macro_aucpr": res["AUCPR"].mean(),
        "weighted_aucpr": np.average(res["AUCPR"], weights=res["test_pos"]),
    }

idle_res, idle_sum = evaluate_ovr_aucpr(train_mix_p, test_idle_p)
active_res, active_sum = evaluate_ovr_aucpr(train_mix_p, test_active_p)
mix_res, mix_sum = evaluate_ovr_aucpr(train_mix_p, test_mix_p)

# =========================================================
# 8) 汇总结果
# =========================================================
summary_table = pd.DataFrame([
    {
        "Scenario": "Train: Mix -> Test: Idle",
        "Macro_AP": idle_sum["macro_ap"],
        "Macro_AUCPR": idle_sum["macro_aucpr"],
        "Weighted_AP": idle_sum["weighted_ap"],
        "Weighted_AUCPR": idle_sum["weighted_aucpr"],
    },
    {
        "Scenario": "Train: Mix -> Test: Active",
        "Macro_AP": active_sum["macro_ap"],
        "Macro_AUCPR": active_sum["macro_aucpr"],
        "Weighted_AP": active_sum["weighted_ap"],
        "Weighted_AUCPR": active_sum["weighted_aucpr"],
    },
    {
        "Scenario": "Train: Mix -> Test: Mix",
        "Macro_AP": mix_sum["macro_ap"],
        "Macro_AUCPR": mix_sum["macro_aucpr"],
        "Weighted_AP": mix_sum["weighted_ap"],
        "Weighted_AUCPR": mix_sum["weighted_aucpr"],
    },
])

print("\n=== AUCPR summary ===")
display(summary_table)

print("\n=== Per-device Idle ===")
display(idle_res)

print("\n=== Per-device Active ===")
display(active_res)

print("\n=== Per-device Mix ===")
display(mix_res)

# =========================================================
# 9) 保存
# =========================================================
out_dir = flow_path.parent
summary_table.to_csv(out_dir / "temporal20_idle80_active_summary.csv", index=False, encoding="utf-8-sig")
idle_res.to_csv(out_dir / "temporal20_idle80_active_idle_per_device.csv", index=False, encoding="utf-8-sig")
active_res.to_csv(out_dir / "temporal20_idle80_active_active_per_device.csv", index=False, encoding="utf-8-sig")
mix_res.to_csv(out_dir / "temporal20_idle80_active_mix_per_device.csv", index=False, encoding="utf-8-sig")

print("\nSaved:")
print(out_dir / "temporal20_idle80_active_summary.csv")

flow_df shape: (87, 16)
device_name
IoT_Modbus          14
IoT_Motion_Light    13
IoT_Fridge          13
IoT_Thermostat      13
IoT_Weather         13
IoT_Garage_Door     12
IoT_GPS_Tracker      9
Name: count, dtype: int64

mode counts:
mode
active    70
idle      17
Name: count, dtype: int64


,device_name,first_time,last_time,mid_time,mode
0,IoT_GPS_Tracker,1.554220e+09,1.554229e+09,1.554225e+09,idle
1,IoT_Fridge,1.554220e+09,1.554230e+09,1.554225e+09,idle
2,IoT_Garage_Door,1.554220e+09,1.554230e+09,1.554225e+09,idle
3,IoT_Motion_Light,1.554220e+09,1.554230e+09,1.554225e+09,idle
4,IoT_Thermostat,1.554220e+09,1.554230e+09,1.554225e+09,idle
5,IoT_Weather,1.554220e+09,1.554230e+09,1.554225e+09,idle
6,IoT_Modbus,1.554220e+09,1.554230e+09,1.554225e+09,idle
7,IoT_Fridge,1.554230e+09,1.554240e+09,1.554235e+09,idle
8,IoT_Modbus,1.554230e+09,1.554241e+09,1.554236e+09,idle
9,IoT_Motion_Light,1.554230e+09,1.554241e+09,1.554236e+09,idle


,device_name,first_time,last_time,mid_time,mode
77,IoT_Motion_Light,1.554335e+09,1.554347e+09,1.554341e+09,active
78,IoT_Modbus,1.554335e+09,1.554347e+09,1.554341e+09,active
79,IoT_Weather,1.554336e+09,1.554347e+09,1.554341e+09,active
80,IoT_Fridge,1.554343e+09,1.554343e+09,1.554343e+09,active
81,IoT_Fridge,1.554351e+09,1.554352e+09,1.554351e+09,active
82,IoT_Garage_Door,1.554352e+09,1.554352e+09,1.554352e+09,active
83,IoT_Thermostat,1.554347e+09,1.554358e+09,1.554353e+09,active
84,IoT_Motion_Light,1.554347e+09,1.554358e+09,1.554353e+09,active
85,IoT_Modbus,1.554347e+09,1.554358e+09,1.554353e+09,active
86,IoT_Weather,1.554347e+09,1.554358e+09,1.554353e+09,active



train_mix shape: (60, 18)
test_idle shape: (6, 18)
test_active shape: (21, 18)
test_mix shape: (27, 18)

train_mix class counts:
device_name
IoT_Fridge          10
IoT_Modbus          10
IoT_Motion_Light     9
IoT_Weather          9
IoT_Thermostat       8
IoT_Garage_Door      7
IoT_GPS_Tracker      7
Name: count, dtype: int64

test_idle class counts:
device_name
IoT_Garage_Door     2
IoT_Thermostat      1
IoT_GPS_Tracker     1
IoT_Weather         1
IoT_Motion_Light    1
Name: count, dtype: int64

test_active class counts:
device_name
IoT_Thermostat      4
IoT_Modbus          4
IoT_Garage_Door     3
IoT_Motion_Light    3
IoT_Weather         3
IoT_Fridge          3
IoT_GPS_Tracker     1
Name: count, dtype: int64
[LightGBM] [Info] Number of positive: 7, number of negative: 53
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000025 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 62
[LightGBM] [Info] Num

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


...


=== Per-device Active ===


,device_name,AP,AUCPR,train_pos,test_pos
6,IoT_Weather,1.000000,1.000000,9,3
5,IoT_Thermostat,1.000000,1.000000,8,4
3,IoT_Modbus,0.797619,0.802656,10,4
4,IoT_Motion_Light,0.333333,0.666667,9,3
1,IoT_GPS_Tracker,0.250000,0.625000,7,1
2,IoT_Garage_Door,0.331349,0.268849,7,3
0,IoT_Fridge,0.333333,0.166667,10,3



=== Per-device Mix ===


,device_name,AP,AUCPR,train_pos,test_pos
5,IoT_Thermostat,1.000000,1.000000,8,5
6,IoT_Weather,0.916667,0.908333,9,4
3,IoT_Modbus,0.788462,0.794231,10,4
1,IoT_GPS_Tracker,0.400000,0.700000,7,2
4,IoT_Motion_Light,0.340909,0.582955,9,4
2,IoT_Garage_Door,0.369719,0.324264,7,5
0,IoT_Fridge,0.250000,0.125000,10,3



Saved:
/content/drive/MyDrive/iot/iot device name/network/packet_csvs_device_only/temporal20_idle80_active_summary.csv
